# Task 2 — SOTA retrieval + Qwen3.5-2B qua vLLM

Pipeline cố định retrieval tốt nhất hiện tại: hybrid `alpha=0.7` → zero-shot `AITeamVN/Vietnamese_Reranker` với `beta=0.6` → top 3 chunks. Qwen chạy ở chế độ non-thinking và copy-edit bản rule-based để hạn chế paraphrase làm giảm METEOR.

Chạy trên runtime mới. Notebook đo 200 câu validation trước, sau đó mới sinh submission public.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = 'https://github.com/caubenq9999/LawRetrieval.git'
BRANCH = 'feat/qa_baseline'
ROOT = Path('/content/LawRetrieval')
DRIVE_ROOT = Path('/content/drive/MyDrive/DSC2026/task2_qa')
INPUT = DRIVE_ROOT / 'input'
CACHE = DRIVE_ROOT / 'artifacts'
QA_DIR = Path('/content/task2_qa')
WORK = Path('/content/task2_work')
WORK.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

In [ ]:
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(ROOT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

assert (ROOT / 'Retrieval-LegalIR/qa_generate_vllm.py').exists(), (
    'Branch chưa có code vLLM mới; hãy commit/push trước.')
subprocess.run(['git', '-C', str(ROOT), 'log', '-1', '--oneline'], check=True)

## Khôi phục đúng corpus/index/embedding Task 2

In [ ]:
chunk_candidates = [CACHE / 'chunks_qa_full.jsonl', CACHE / 'chunks_qa_full',
                    CACHE / 'chunks_qa.jsonl']
cached_chunks = next((p for p in chunk_candidates if p.exists()), None)
required = [INPUT / 'train.json', INPUT / 'public-official.json',
            INPUT / 'selected-contexts.zip', CACHE / 'index_qa.zip',
            CACHE / 'emb_qa_v2.zip']
missing = [str(p) for p in required if not p.exists()]
assert not missing and cached_chunks, 'Thiếu artifact:\n' + '\n'.join(missing)

QA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(INPUT / 'train.json', QA_DIR / 'train.json')
shutil.copy2(INPUT / 'public-official.json', QA_DIR / 'public-official.json')
context_dir = QA_DIR / 'selected-contexts'
if len(list(context_dir.glob('context_*.json'))) != 8532:
    subprocess.run(['unzip', '-q', '-o', str(INPUT / 'selected-contexts.zip'),
                    '-d', str(QA_DIR)], check=True)

chunks = WORK / 'chunks_qa.jsonl'
index_dir = WORK / 'index_qa'
emb_dir = WORK / 'emb_qa_v2'
if not chunks.exists(): shutil.copy2(cached_chunks, chunks)
if not (index_dir / 'meta.json').exists():
    subprocess.run(['unzip', '-q', '-o', str(CACHE / 'index_qa.zip'), '-d', str(WORK)], check=True)
if not (emb_dir / 'meta.json').exists():
    subprocess.run(['unzip', '-q', '-o', str(CACHE / 'emb_qa_v2.zip'), '-d', str(WORK)], check=True)

imeta = json.loads((index_dir / 'meta.json').read_text())
emeta = json.loads((emb_dir / 'meta.json').read_text())
imeta['chunks_path'] = str(chunks.resolve())
(index_dir / 'meta.json').write_text(json.dumps(imeta, indent=2), encoding='utf-8')
with open(chunks, 'rb') as f: chunk_rows = sum(1 for _ in f)
assert chunk_rows == imeta['n_chunks'] == emeta['n'] == 487194
print('READY:', chunk_rows, 'rows')

In [ ]:
def run_live(command, cwd):
    process = subprocess.Popen(command, cwd=str(cwd), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code != 0: raise RuntimeError(f'Exit code {code}')

qa_predict = ROOT / 'Retrieval-LegalIR/qa_predict.py'
qa_generate = ROOT / 'Retrieval-LegalIR/qa_generate_vllm.py'
RERANKER = 'AITeamVN/Vietnamese_Reranker'
MODEL = 'Qwen/Qwen3.5-2B'

## 1. Retrieve 200 câu validation bằng cấu hình SOTA

In [ ]:
eval_candidates = WORK / 'sota_contexts_eval200.json'
run_live([
    sys.executable, '-u', str(qa_predict), '--qa-dir', str(QA_DIR),
    '--index', str(index_dir), '--emb', str(emb_dir),
    '--retriever', 'hybrid', '--alpha', '0.7',
    '--reranker', RERANKER, '--beta', '0.6', '--batch', '128',
    '--eval', '--offset', '200', '-n', '200',
    '--nchunk', '3', '--style', 'cite',
    '--candidates-out', str(eval_candidates)
], ROOT / 'Retrieval-LegalIR')

## 2. vLLM sinh đáp án validation
Kết quả cuối cell tự so rule và vLLM trên cùng context. Mốc cần vượt là 0.5347.

In [ ]:
eval_out = WORK / 'submission_vllm_eval200'
run_live([
    sys.executable, '-u', str(qa_generate),
    '--candidates', str(eval_candidates), '--out', str(eval_out),
    '--model', MODEL, '--max-model-len', '8192', '--max-tokens', '1200',
    '--gpu-memory-utilization', '0.85', '--temperature', '0.0'
], ROOT / 'Retrieval-LegalIR')

## 3. Sinh public submission
Có thể đổi `RUN_PUBLIC=False` nếu validation tụt quá sâu.

In [ ]:
RUN_PUBLIC = True
if RUN_PUBLIC:
    public_candidates = WORK / 'sota_contexts_public.json'
    rule_prefix = WORK / 'submission_rule_sota'
    run_live([
        sys.executable, '-u', str(qa_predict), '--qa-dir', str(QA_DIR),
        '--questions', str(QA_DIR / 'public-official.json'),
        '--index', str(index_dir), '--emb', str(emb_dir),
        '--retriever', 'hybrid', '--alpha', '0.7',
        '--reranker', RERANKER, '--beta', '0.6', '--batch', '128',
        '--nchunk', '3', '--style', 'cite', '--out', str(rule_prefix),
        '--candidates-out', str(public_candidates)
    ], ROOT / 'Retrieval-LegalIR')

    public_out = WORK / 'submission_qa_vllm_qwen35_2b'
    run_live([
        sys.executable, '-u', str(qa_generate),
        '--candidates', str(public_candidates), '--out', str(public_out),
        '--model', MODEL, '--max-model-len', '8192', '--max-tokens', '1200',
        '--gpu-memory-utilization', '0.85', '--temperature', '0.0'
    ], ROOT / 'Retrieval-LegalIR')
    shutil.copy2(str(public_out) + '.zip', CACHE / (public_out.name + '.zip'))
    print('SUBMISSION:', str(public_out) + '.zip')